<a href="https://colab.research.google.com/github/MohdFuzailHaider/flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Loading Dataset

In [18]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

def load_month(month):
    path = hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        repo_type="dataset",
        filename=f"fact_content_daily_performance/month={month}/data_0.parquet",
        token=HF_TOKEN,)

    return pd.read_parquet(path)
mar = load_month("2026-03")

print(mar.shape)

(9841378, 30)


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1: Refreshing strong content can be valuable**
* One finding in the flyrank research paper is that freshness amplifies quality rather than replacing it, and that refreshing strong content can be more useful than simply refreshing weak content.
* The paper compares content across age and freshness groups and reports differences in its portfolio-level health score.

**My Methodology Question**
* How confidently can the observed differences be attributed to refreshing itself, rather than differences that already existed between the pages that were refreshed and the pages that were not?
* The paper uses observational portfolio data, so this question is not a criticism of the finding/ It asks whether the comparison design can support a causal claim or whether the evidence is better interpreted as a directional association.
* The paper itself discloses that the study is observational and that correlations do not prove causation.
* A stronger validation design could compare similar pages before and after a refresh, or compare refreshed and non-refreshed pages while controlling for factors such as topic, starting performance, and content quality.


---


**Finding 2: AI-generated content is not penalized by default**
* The paper reports that, within its mostly AI-authored portfolio, age-controlled model cohorts did not show a simple blanket penalty tied only to AI use.
* Instead, the paper suggests that differences may depend more on factors such as model choice, editing standards, process quality, and topic fit.

**My Methodology question**
* Does comparing content within age tiers sufficiently control for other differences that could affect performance, such as topic, publication timing, editing standards and the workflow to produce the content?
* This is a constructive question about the validation design. Age control improves the comparison because content age is identified as a potential confounding variable in the paper's methodology.
* However, age may not be the only factor influencing performance.
* Therefore, the finding is most safely interpreted as:
    Within this portfolio, the observed evidence did not show a simple blanket penalty associated only with AI use.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
mar["report_date"] = pd.to_datetime(mar["report_date"])

early = mar[(mar["report_date"] >= "2026-03-01") &
    (mar["report_date"] <= "2026-03-15")].copy()

late = mar[(mar["report_date"] >= "2026-03-16") &
    (mar["report_date"] <= "2026-03-31")].copy()

print("Early shape:", early.shape)
print("Late shape:", late.shape)
early_features = (
    early.groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        pageviews=("ga4_pageviews", "sum"),
        sessions=("ga4_sessions", "sum"),
        users=("ga4_users", "sum"),
        engaged_sessions=("ga4_engaged_sessions", "sum"),
        engagement_sec=("ga4_total_engagement_sec", "sum"),
        organic_sessions=("sessions_organic", "sum"),
        scroll_events=("scroll_events", "sum"),
    )
)
late_features = (
    late.groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        late_impressions=("gsc_impressions", "sum"),
        late_clicks=("gsc_clicks", "sum"),
        late_avg_position=("gsc_avg_position", "mean"),
        late_pageviews=("ga4_pageviews", "sum"),
        late_sessions=("ga4_sessions", "sum"),
        late_users=("ga4_users", "sum"),
        late_engaged_sessions=("ga4_engaged_sessions", "sum"),
        late_engagement_sec=("ga4_total_engagement_sec", "sum"),
        late_organic_sessions=("sessions_organic", "sum"),
        late_scroll_events=("scroll_events", "sum"),
    )
)
model_df = early_features.merge(
    late_features[
        ["client_hash_id", "content_hash_id", "late_impressions"]],
    on=["client_hash_id", "content_hash_id"],how="left")

# Pages missing from the late period had no recorded late impressions
model_df["late_impressions"] = model_df["late_impressions"].fillna(0)

model_df["early_daily_impressions"] = model_df["impressions"] / 15
model_df["late_daily_impressions"] = model_df["late_impressions"] / 16

model_df["is_declining"] = (
    (model_df["impressions"] > 0) &
    (model_df["late_daily_impressions"] < model_df["early_daily_impressions"])
).astype(int)

Early shape: (4642255, 30)
Late shape: (5199123, 30)


In [ ]:
feature_cols = ["impressions", "clicks", "avg_position", "pageviews",
    "sessions", "users", "engaged_sessions", "engagement_sec",
    "organic_sessions","scroll_events"]

X = model_df[feature_cols]
y = model_df["is_declining"]
groups = model_df["client_hash_id"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of clients:", groups.nunique())
print("\nDecline rate:", y.mean())

X shape: (319759, 10)
y shape: (319759,)
Number of clients: 52

Decline rate: 0.2361309611301011


In [ ]:
from sklearn.model_selection import train_test_split

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y)

print("BEFORE — Random Row Split")
print("Training shape:", X_train_random.shape)
print("Testing shape:", X_test_random.shape)

BEFORE — Random Row Split
Training shape: (255807, 10)
Testing shape: (63952, 10)


In [ ]:
from sklearn.impute import SimpleImputer

imputer_random = SimpleImputer(strategy="median")

X_train_random_imputed = imputer_random.fit_transform(X_train_random)
X_test_random_imputed = imputer_random.transform(X_test_random)

print("Missing values in training after imputation:",
      pd.DataFrame(X_train_random_imputed).isna().sum().sum())

print("Missing values in testing after imputation:",
      pd.DataFrame(X_test_random_imputed).isna().sum().sum())

Missing values in training after imputation: 0
Missing values in testing after imputation: 0


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_random = RandomForestClassifier(
    n_estimators=200, max_depth=15,
    min_samples_leaf=10, random_state=42,
    n_jobs=-1)

rf_random.fit(X_train_random_imputed, y_train_random)

print("Random-split model trained successfully!")

Random-split model trained successfully!


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

random_pred = rf_random.predict(X_test_random_imputed)
random_prob = rf_random.predict_proba(X_test_random_imputed)[:, 1]

random_accuracy = accuracy_score(y_test_random, random_pred)
random_precision = precision_score(y_test_random, random_pred)
random_recall = recall_score(y_test_random, random_pred)
random_f1 = f1_score(y_test_random, random_pred)
random_auc = roc_auc_score(y_test_random, random_prob)

print("BEFORE — Random Row Split")
print("Accuracy:", random_accuracy)
print("Precision:", random_precision)
print("Recall:", random_recall)
print("F1 Score:", random_f1)
print("ROC-AUC:", random_auc)

BEFORE — Random Row Split
Accuracy: 0.8077464348261196
Precision: 0.5874797356278838
Recall: 0.6239321899211973
F1 Score: 0.605157519509297
ROC-AUC: 0.8853593218678711


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("AFTER — Grouped-by-Client Split")
print("Training shape:", X_train_group.shape)
print("Testing shape:", X_test_group.shape)

print("\nTraining clients:", groups_train.nunique())
print("Testing clients:", groups_test.nunique())

overlap = set(groups_train.unique()) & set(groups_test.unique())

print("Client overlap:", len(overlap))

AFTER — Grouped-by-Client Split
Training shape: (265352, 10)
Testing shape: (54407, 10)

Training clients: 41
Testing clients: 11
Client overlap: 0


In [ ]:
from sklearn.impute import SimpleImputer

imputer_group = SimpleImputer(strategy="median")

X_train_group_imputed = imputer_group.fit_transform(X_train_group)
X_test_group_imputed = imputer_group.transform(X_test_group)

print(
    "Missing values in grouped training after imputation:",
    pd.DataFrame(X_train_group_imputed).isna().sum().sum()
)

print(
    "Missing values in grouped testing after imputation:",
    pd.DataFrame(X_test_group_imputed).isna().sum().sum()
)

Missing values in grouped training after imputation: 0
Missing values in grouped testing after imputation: 0


In [ ]:
rf_group = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

rf_group.fit(X_train_group_imputed, y_train_group)

print("Grouped-split model trained successfully!")

Grouped-split model trained successfully!


In [ ]:
group_pred = rf_group.predict(X_test_group_imputed)
group_prob = rf_group.predict_proba(X_test_group_imputed)[:, 1]

group_accuracy = accuracy_score(y_test_group, group_pred)
group_precision = precision_score(y_test_group, group_pred)
group_recall = recall_score(y_test_group, group_pred)
group_f1 = f1_score(y_test_group, group_pred)
group_auc = roc_auc_score(y_test_group, group_prob)

print("AFTER — Grouped-by-Client Split")
print("Accuracy:", group_accuracy)
print("Precision:", group_precision)
print("Recall:", group_recall)
print("F1 Score:", group_f1)
print("ROC-AUC:", group_auc)

AFTER — Grouped-by-Client Split
Accuracy: 0.7733747495726653
Precision: 0.42770367587200203
Recall: 0.6815687057179922
F1 Score: 0.5255867641400539
ROC-AUC: 0.8462457065278941


In [ ]:
comparison = pd.DataFrame({
    "Metric": ["Accuracy","Precision","Recall","F1 Score","ROC-AUC"],

    "Before: Random Row Split": [random_accuracy,random_precision,random_recall,random_f1,random_auc],

    "After: Grouped-by-Client Split": [group_accuracy,group_precision,group_recall,group_f1,group_auc]
})

comparison["Difference"] = (
    comparison["After: Grouped-by-Client Split"] - comparison["Before: Random Row Split"]
)

comparison

,Metric,Before: Random Row Split,After: Grouped-by-Client Split,Difference
0,Accuracy,0.807746,0.773375,-0.034372
1,Precision,0.587480,0.427704,-0.159776
2,Recall,0.623932,0.681569,0.057637
3,F1 Score,0.605158,0.525587,-0.079571
4,ROC-AUC,0.885359,0.846246,-0.039114


**Before/After Validation Comparison**
* The model was evaluated under two different validation designs. First, I used a random row split. Under this design, content records from the same client could appear in both the training and testing sets.
* I then evaluated the same model using a grouped-by-client split, where clients in the test set were completely absent from the training set. The grouped split had zero client overlap between training and testing.
* Under the random row split, the model measured a ROC-AUC of approximately 0.885 and an F1 score of approximately 0.605. Under the grouped-by-client split, the measured ROC-AUC was approximately 0.846 and the F1 score was approximately 0.526
* This comparrison suggests that the random row split produced more optimistic performance estimates for this dataset. When evaluated on completely unseen clients, measured performance decreased, particularly for precision and F1 score.
* The grouped-by-client results provide a more relevant estimate of how this model may generalize to content from clients that were not represented during training. This is directional validation finding rather than evidence that th emodel will perform at exactly these levels for every future client.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
print("Final model features:")
print(feature_cols)

print("\nNumber of features:", len(feature_cols))

In [ ]:
future_target_cols = ["late_impressions","late_daily_impressions","is_declining"]

leakage_check = pd.DataFrame({"Column": future_target_cols,
    "Used as Model Feature": [col in feature_cols for col in future_target_cols]
})

leakage_check

,Column,Used as Model Feature
0,late_impressions,False
1,late_daily_impressions,False
2,is_declining,False


In [ ]:
target_relationship_check = pd.DataFrame({
    "Column": [
        "impressions",
        "early_daily_impressions",
        "late_impressions",
        "late_daily_impressions",
        "is_declining"
    ],

    "Used as Model Feature": [
        "impressions" in feature_cols,
        "early_daily_impressions" in feature_cols,
        "late_impressions" in feature_cols,
        "late_daily_impressions" in feature_cols,
        "is_declining" in feature_cols
    ],

    "Role": [
        "Early-period model feature",
        "Used in target construction",
        "Future-period outcome",
        "Used in target construction",
        "Target variable"
    ]
})

target_relationship_check

,Column,Used as Model Feature,Role
0,impressions,True,Early-period model feature
1,early_daily_impressions,False,Used in target construction
2,late_impressions,False,Future-period outcome
3,late_daily_impressions,False,Used in target construction
4,is_declining,False,Target variable


In [ ]:
print("Features used in the model:")
print(X.columns.tolist())

print("\nAre these exactly the intended features?")
print(set(X.columns) == set(feature_cols))

Features used in the model:
['impressions', 'clicks', 'avg_position', 'pageviews', 'sessions', 'users', 'engaged_sessions', 'engagement_sec', 'organic_sessions', 'scroll_events']

Are these exactly the intended features?
True


I audited the final feature set used by the model. The model uses ten features:

- impressions
- clicks
- avg_position
- pageviews
- sessions
- users
- engaged_sessions
- engagement_sec
- organic_sessions
- scroll_events

A programmatic check confirmed that the model used exactly these intended features.

I also checked whether future-period or target-related columns were accidentally included as model features. `late_impressions`, `late_daily_impressions`, and `is_declining` were all confirmed to be absent from the final feature set.

The model features were constructed from the observation period of **March 1–15, 2026**, while impressions from **March 16–31, 2026** were used to construct the outcome label. Based on this audit, I did not identify direct future-period information leakage into the final model features.

One relationship should still be disclosed: `impressions` is used as a model feature, while early-period impressions also contribute to the construction of the decline label through comparison with later-period impressions. This is not future leakage because early-period information is available before the outcome period. However, this dependency means that the relationship between `impressions` and the target should be interpreted carefully.

### Data availability limitation

I also audited the warehouse availability flags. The March data showed that GA4 availability was incomplete for some clients and observation-period rows. The model includes several GA4-derived features: `pageviews`, `sessions`, `users`, `engaged_sessions`, and `engagement_sec`.

The original feature pipeline aggregated these GA4 metrics without explicitly filtering rows where `ga4_data_available` was `False`. Therefore, some aggregated GA4 feature values may include zero-filled unavailable periods rather than representing only genuine zero engagement.

This is a **data availability and measurement limitation**, rather than direct future-period feature leakage. The grouped-by-client validation results are therefore interpreted with this limitation in mind.

Overall, the audit did not identify direct future-period leakage into the final feature set, but it identified two important limitations that should be disclosed:

1. Early-period `impressions` is related to the construction of the decline label.
2. GA4-derived features may be affected by incomplete data availability during the observation period.

These results support using the model as a **directional decision-support tool**, rather than treating its predictions as definitive or fully generalizable.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim**

> "The model accurately predicts which pages will decline and can identify pages that should be refreshed."

**Revised claim**

The validated model **flags and ranks pages by their estimated likelihood of meeting the decline label defined in this analysis**.

Under grouped-by-client validation, the model measured:

- Accuracy: approximately **0.773**
- Precision: approximately **0.427**
- Recall: approximately **0.682**
- F1 score: approximately **0.526**
- ROC-AUC: approximately **0.846**

These results were measured using a grouped-by-client split, where clients in the test set were not included in the training set.

The results are therefore best interpreted as evidence that, **within this dataset and validation design**, the model can provide a directional ranking signal for prioritizing pages for human review.

The model does not prove that a page will decline, and it does not determine that refreshing a page will improve performance. It is intended as a **decision-support tool** for identifying pages that may be worth reviewing first.

The results should also be interpreted alongside the data availability limitation identified in the leakage audit, particularly the incomplete availability of some GA4-derived measurements.

## Self-check

Before you submit, confirm each line honestly:

- [ YES ] Every section above is filled — markdown thinking AND the code that backs it
- [ YES ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ YES ] No client names, URLs, or private queries anywhere
- [ YES ] My claims use careful words: observed, measured, directional, decision-support
- [ YES ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.